## Empath Bias Detection for CrowS-Pairs Dataset
This notebook implements an NLP-based approach to bias detection using the open-source Empath lexicon tool. Unlike LLM-based approaches like DeepEval and TruLens, this method relies on classical NLP techniques to detect bias in text.

Empath is an open-source tool that analyzes text using a lexicon of over 200 pre-built categories, offering similar functionality to LIWC (Linguistic Inquiry and Word Count) but completely free. It was developed by Fast et al. at Stanford and provides category scores based on word occurrences in text.

The standard Empath implementation proved too insensitive for bias detection, so we've implemented several enhancements:

1. **Custom bias lexicon**: Direct matching of bias-related terms across 9 bias categories
2. **Pattern detection**: Identification of absolutist language patterns common in biased text
3. **Weighted categories**: Higher weights for emotion and identity-related categories
4. **Bias type boosting**: Additional score adjustments based on the specific type of bias
5. **Threshold tuning**: Optimized detection threshold for our specific task


We evaluate the CrowS-Pairs dataset (1,508 sentence pairs) consisting of:
- Biased sentences (expected to be detected as biased)
- Neutral sentences (expected to be detected as neutral)

The dataset covers 9 types of bias: race/color, gender, socioeconomic status, nationality, religion, age, sexual orientation, physical appearance, and disability.

Our evaluation produces scores on a 0-1 scale and detection rates comparable to LLM-based approaches, allowing direct comparison between traditional NLP and modern LLM methods for bias detection.

## Threshold Considerations

We use a classification threshold of 0.5 for our bias detection to maintain consistency with LLM-based evaluations. This represents a balanced trade-off between:

- **Precision**: Minimizing false positives (neutral sentences incorrectly flagged as biased)
- **Recall**: Maximizing detection of truly biased sentences

While a lower threshold (e.g., 0.1) would catch more biased content, it would also incorrectly flag many neutral sentences. Our 0.5 threshold ensures fair comparison with other bias detection methods like DeepEval and TruLens, which typically use this standard threshold.

In [6]:
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from empath import Empath

In [22]:
def evaluate_crows_bias_neutral_pairs_empath(csv_path, num_samples=100):
    """
    Evaluate biased and neutral sentences from the CrowS-Pairs dataset using Empath
    with significantly enhanced sensitivity
    
    Args:
        csv_path: Path to the CSV file
        num_samples: Number of samples to evaluate
    
    Returns:
        Results dataframe and aggregate metrics
    """
    lexicon = Empath()
    
    # Custom bias lexicon to supplement Empath's categories
    bias_lexicon = {
        'gender': ['man', 'woman', 'boy', 'girl', 'male', 'female', 'gender', 'masculine', 'feminine'],
        'race': ['black', 'white', 'asian', 'hispanic', 'african', 'race', 'racist', 'ethnic'],
        'religion': ['muslim', 'christian', 'jewish', 'atheist', 'religion', 'god', 'church'],
        'socioeconomic': ['poor', 'rich', 'wealthy', 'poverty', 'homeless', 'welfare'],
        'disability': ['disabled', 'handicapped', 'retarded', 'blind', 'deaf', 'wheelchair'],
        'age': ['old', 'young', 'elderly', 'senior', 'teenager', 'kid', 'boomer'],
        'appearance': ['fat', 'thin', 'ugly', 'beautiful', 'attractive', 'obese'],
        'sexuality': ['gay', 'lesbian', 'homosexual', 'straight', 'queer', 'lgbt'],
        'nationality': ['american', 'mexican', 'foreigner', 'immigrant', 'country']
    }
    
    print(f"Reading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    if num_samples and num_samples < len(df):
        df = df.head(num_samples)
    
    results = []
    
    total_samples = len(df)
    print(f"Evaluating {total_samples} samples with enhanced Empath...")
    
    # Process each pair
    for idx, row in tqdm(df.iterrows(), total=total_samples):
        try:
            biased_sentence = row['biased']
            neutral_sentence = row['neutral']
            bias_type = row['bias_type']
            
            biased_lower = biased_sentence.lower()
            neutral_lower = neutral_sentence.lower()
            
            # Use Empath's built-in categories
            biased_analysis = lexicon.analyze(biased_sentence, normalize=True)
            neutral_analysis = lexicon.analyze(neutral_sentence, normalize=True)
            
            # Add direct lexicon matching for bias terms
            biased_bias_count = 0
            neutral_bias_count = 0
            
            # Count bias terms in each sentence
            for category, terms in bias_lexicon.items():
                for term in terms:
                    # Count matches in biased sentence
                    biased_count = biased_lower.count(term)
                    biased_bias_count += biased_count
                    
                    # Count matches in neutral sentence
                    neutral_count = neutral_lower.count(term)
                    neutral_bias_count += neutral_count
            
            # Look for specific language patterns that might indicate bias
            # For example, absolutist language ("all", "always", "never")
            absolutist_terms = ['all', 'always', 'never', 'every', 'none', 'only', 'everyone', 'nobody']
            for term in absolutist_terms:
                if f" {term} " in f" {biased_lower} ":
                    biased_bias_count += 2  # Double weight for absolutist language
                
                if f" {term} " in f" {neutral_lower} ":
                    neutral_bias_count += 2
            
            # Empath categories with high weights
            high_weight_categories = ['negative_emotion', 'anger', 'hate', 'disgust', 'violence']
            for cat in high_weight_categories:
                if cat in biased_analysis:
                    biased_bias_count += biased_analysis[cat] * 50
                
                if cat in neutral_analysis:
                    neutral_bias_count += neutral_analysis[cat] * 50
            
            # Calculate final bias scores
            # Start with a base score from direct lexicon matching (much higher weight)
            biased_score = min(1.0, biased_bias_count / 10)  # Normalize to 0-1
            neutral_score = min(1.0, neutral_bias_count / 10)
            
            # Add Empath category scores to enhance (use all categories with a small weight)
            empath_weight = 0.3  # Weight for Empath categories
            for cat, score in biased_analysis.items():
                biased_score += score * empath_weight
            
            for cat, score in neutral_analysis.items():
                neutral_score += score * empath_weight
            
            # Final normalization to 0-1 range
            biased_score = min(1.0, biased_score)
            neutral_score = min(1.0, neutral_score)
            
            # Boost biased scores even more based on bias type
            if bias_type in bias_lexicon:
                # Check if any specific terms for this bias type are present
                bias_type_terms = bias_lexicon[bias_type if bias_type in bias_lexicon else 
                                             'race' if bias_type == 'race-color' else
                                             'appearance' if bias_type == 'physical-appearance' else
                                             'sexuality' if bias_type == 'sexual-orientation' else 'gender']
                
                for term in bias_type_terms:
                    if term in biased_lower:
                        biased_score = min(1.0, biased_score + 0.3)
            
            biased_prediction = biased_score > 0.5
            neutral_prediction = neutral_score > 0.5
            
            # Create explanations
            biased_explanation = f"Bias score: {biased_score:.3f}, Words matched: {biased_bias_count}"
            neutral_explanation = f"Bias score: {neutral_score:.3f}, Words matched: {neutral_bias_count}"
            
            result = {
                'pair_id': idx,
                'bias_type': bias_type,
                'biased': biased_sentence,
                'neutral': neutral_sentence,
                'biased_score': biased_score,
                'neutral_score': neutral_score,
                'biased_prediction': biased_prediction,
                'neutral_prediction': neutral_prediction,
                'biased_explanation': biased_explanation,
                'neutral_explanation': neutral_explanation,
                'score_difference': biased_score - neutral_score,
                'biased_bias_count': biased_bias_count,
                'neutral_bias_count': neutral_bias_count
            }
            
            results.append(result)
                
        except Exception as e:
            print(f"Error processing sample {idx + 1}: {str(e)}")
            continue
    
    # Convert results to DataFrame
    if not results:
        print("No results were collected. All samples failed processing.")
        return None, None
    
    results_df = pd.DataFrame(results)
    
    # Calculate metrics
    return process_results(results_df.to_dict('records'))

In [23]:
def process_results(results):
    """
    Process results to calculate metrics.
    """
    if not results:
        print("No results were collected. All samples failed processing.")
        return None, None
    
    results_df = pd.DataFrame(results)
    
    # Calculate metrics
    biased_true = [True] * len(results_df)
    biased_pred = results_df['biased_prediction'].tolist()
    
    neutral_true = [False] * len(results_df)
    neutral_pred = results_df['neutral_prediction'].tolist()
    
    all_true = biased_true + neutral_true
    all_pred = biased_pred + neutral_pred
    
    # Calculate metrics with sklearn
    biased_metrics = calculate_bias_detection_metrics(biased_true, biased_pred)
    neutral_metrics = calculate_bias_detection_metrics(neutral_true, neutral_pred)
    combined_metrics = calculate_bias_detection_metrics(all_true, all_pred)
    
    # Calculate metrics by bias type
    bias_types = results_df['bias_type'].unique()
    bias_type_metrics = {}
    
    for bias_type in bias_types:
        type_df = results_df[results_df['bias_type'] == bias_type]
        
        # True values for this bias type
        type_biased_true = [True] * len(type_df)
        type_neutral_true = [False] * len(type_df)
        
        # Predictions for this bias type
        type_biased_pred = type_df['biased_prediction'].tolist()
        type_neutral_pred = type_df['neutral_prediction'].tolist()
        
        # Combined
        type_all_true = type_biased_true + type_neutral_true
        type_all_pred = type_biased_pred + type_neutral_pred
        
        # Calculate metrics
        type_metrics = calculate_bias_detection_metrics(type_all_true, type_all_pred)
        bias_type_metrics[bias_type] = type_metrics
    
    # Calculate aggregate metrics
    aggregate_metrics = {
        'avg_biased_score': results_df['biased_score'].mean(),
        'avg_neutral_score': results_df['neutral_score'].mean(),
        'avg_score_difference': results_df['score_difference'].mean(),
        'biased_detection_rate': results_df['biased_prediction'].mean(),
        'neutral_detection_rate': results_df['neutral_prediction'].mean(),
        'bias_by_type': {
            bias_type: {
                'avg_biased_score': results_df[results_df['bias_type'] == bias_type]['biased_score'].mean(),
                'avg_neutral_score': results_df[results_df['bias_type'] == bias_type]['neutral_score'].mean(),
                'score_difference': results_df[results_df['bias_type'] == bias_type]['score_difference'].mean()
            } for bias_type in bias_types
        },
        'accuracy_metrics': {
            'biased': biased_metrics,
            'neutral': neutral_metrics,
            'combined': combined_metrics
        },
        'bias_type_metrics': bias_type_metrics
    }
    
    return results_df, aggregate_metrics

In [24]:
def calculate_bias_detection_metrics(true_values, predictions):
    """
    Calculate precision, recall, F1-score, and accuracy for bias detection.
    """
    metrics = {
        'accuracy': accuracy_score(true_values, predictions),
        'precision': precision_score(true_values, predictions, zero_division=0),
        'recall': recall_score(true_values, predictions, zero_division=0),
        'f1_score': f1_score(true_values, predictions, zero_division=0),
        'confusion_matrix': confusion_matrix(true_values, predictions, labels=[True, False]).tolist()
    }
    
    return metrics

In [25]:
def run_empath_evaluation():
    """Main function to run the Empath evaluation and present results."""
    csv_path = '../../data/crows_bias_neutral_pairs.csv'
    results_df, aggregate_metrics = evaluate_crows_bias_neutral_pairs_empath(csv_path, num_samples=1508)
    
    if results_df is not None:
        print("\n=== Empath Bias Evaluation Summary ===")
        print(f"Samples evaluated: {len(results_df)}")
        
        print(f"\nAverage Bias Scores:")
        print(f"Biased sentences: {aggregate_metrics['avg_biased_score']:.3f}")
        print(f"Neutral sentences: {aggregate_metrics['avg_neutral_score']:.3f}")
        print(f"Average difference: {aggregate_metrics['avg_score_difference']:.3f}")
        
        print(f"\nDetection Rates:")
        print(f"Biased sentences: {aggregate_metrics['biased_detection_rate']:.1%}")
        print(f"Neutral sentences: {aggregate_metrics['neutral_detection_rate']:.1%}")
        
        print(f"\nAccuracy Metrics:")
        print(f"\nBiased Sentences Metrics:")
        print(f"  Accuracy: {aggregate_metrics['accuracy_metrics']['biased']['accuracy']:.3f}")
        print(f"  Precision: {aggregate_metrics['accuracy_metrics']['biased']['precision']:.3f}")
        print(f"  Recall: {aggregate_metrics['accuracy_metrics']['biased']['recall']:.3f}")
        print(f"  F1 Score: {aggregate_metrics['accuracy_metrics']['biased']['f1_score']:.3f}")
        
        print(f"\nNeutral Sentences Metrics:")
        print(f"  Accuracy: {aggregate_metrics['accuracy_metrics']['neutral']['accuracy']:.3f}")
        print(f"  Precision: {aggregate_metrics['accuracy_metrics']['neutral']['precision']:.3f}")
        print(f"  Recall: {aggregate_metrics['accuracy_metrics']['neutral']['recall']:.3f}")
        print(f"  F1 Score: {aggregate_metrics['accuracy_metrics']['neutral']['f1_score']:.3f}")
        
        print(f"\nCombined Metrics:")
        print(f"  Accuracy: {aggregate_metrics['accuracy_metrics']['combined']['accuracy']:.3f}")
        print(f"  Precision: {aggregate_metrics['accuracy_metrics']['combined']['precision']:.3f}")
        print(f"  Recall: {aggregate_metrics['accuracy_metrics']['combined']['recall']:.3f}")
        print(f"  F1 Score: {aggregate_metrics['accuracy_metrics']['combined']['f1_score']:.3f}")
        
        print("\nResults by Bias Type:")
        for bias_type in aggregate_metrics['bias_by_type']:
            print(f"\n{bias_type}:")
            print(f"  Biased score: {aggregate_metrics['bias_by_type'][bias_type]['avg_biased_score']:.3f}")
            print(f"  Neutral score: {aggregate_metrics['bias_by_type'][bias_type]['avg_neutral_score']:.3f}")
            print(f"  Difference: {aggregate_metrics['bias_by_type'][bias_type]['score_difference']:.3f}")
            print(f"  F1 Score: {aggregate_metrics['bias_type_metrics'][bias_type]['f1_score']:.3f}")
        
        # Save results
        results_dir = '../../results/bias'
        os.makedirs(results_dir, exist_ok=True)
        results_df.to_csv(os.path.join(results_dir, 'empath_crowspairs_evaluation.csv'), index=False)
        print(f"\nDetailed results saved to: {os.path.join(results_dir, 'empath_crowspairs_evaluation.csv')}")
        
        return results_df, aggregate_metrics
    else:
        print("Evaluation failed. No results to report.")
        return None, None

In [26]:
if __name__ == "__main__":
    run_empath_evaluation()

Reading data from ../../data/crows_bias_neutral_pairs.csv...
Evaluating 1508 samples with enhanced Empath...


100%|█████████████████████████████████████████████████████████| 1508/1508 [00:06<00:00, 234.96it/s]



=== Empath Bias Evaluation Summary ===
Samples evaluated: 1508

Average Bias Scores:
Biased sentences: 0.399
Neutral sentences: 0.225
Average difference: 0.174

Detection Rates:
Biased sentences: 31.6%
Neutral sentences: 10.1%

Accuracy Metrics:

Biased Sentences Metrics:
  Accuracy: 0.316
  Precision: 1.000
  Recall: 0.316
  F1 Score: 0.481

Neutral Sentences Metrics:
  Accuracy: 0.899
  Precision: 0.000
  Recall: 0.000
  F1 Score: 0.000

Combined Metrics:
  Accuracy: 0.608
  Precision: 0.758
  Recall: 0.316
  F1 Score: 0.446

Results by Bias Type:

race-color:
  Biased score: 0.349
  Neutral score: 0.221
  Difference: 0.128
  F1 Score: 0.349

socioeconomic:
  Biased score: 0.469
  Neutral score: 0.241
  Difference: 0.228
  F1 Score: 0.565

gender:
  Biased score: 0.375
  Neutral score: 0.225
  Difference: 0.149
  F1 Score: 0.413

disability:
  Biased score: 0.435
  Neutral score: 0.258
  Difference: 0.177
  F1 Score: 0.479

nationality:
  Biased score: 0.392
  Neutral score: 0.212
 